### Baseline models XGBoost and Logistic Regression on training data including directors and writers

(!)) Before running, check dependencies and requirements

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root.name != "big_data_assignment" and project_root.parent != project_root:
    project_root = project_root.parent

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Using project root:", project_root)
print("Ensured project root on sys.path")

Using project root: /Users/lisavanoosten/Desktop/Big-Data/big_data_assignment
Ensured project root on sys.path


In [2]:
import sys
!{sys.executable} -m pip install xgboost==2.0.3

In [3]:

from src.utils.config import load_config
from src.data.dataloaders import load_train_data, load_validation_data
from src.pipeline.baseline import run_baselines


# Shared baseline notebook

This notebook runs the **shared baseline models** on the IMDB CSV data using the project source code.

- Uses the central `config/config.yaml` to configure **data paths, feature settings, and hyperparameters**.
- Imports shared helpers from `src/` (config loader, dataloaders, and baseline pipeline).
- Works **only on CSV data referenced in the YAML config**.
- **JSON files or other external datasets are _not_ included here**; those should be handled in separate, member-specific notebooks.


## 1. Environment and imports

This cell:
- Locates the project root folder (named `big_data_assignment`).
- Adds `src/` to `sys.path` so we can import the shared modules.
- (Optionally) enables autoreload for a smoother dev experience inside the notebook.


In [4]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Infer project root by walking up until we find the big_data_assignment folder
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "big_data_assignment" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Project root:", PROJECT_ROOT)
print("Ensured project root on sys.path")


Project root: /Users/lisavanoosten/Desktop/Big-Data/big_data_assignment
Ensured project root on sys.path


## 2. Load config and data helpers

Here we:
- Load the central YAML config via `load_config()`.
- Import the shared dataloaders that operate on the CSV files.
- Import the `run_baselines()` pipeline that trains/evaluates the baseline models.

Hyperparameters (e.g. model type and settings) are controlled from `config/config.yaml` under the `model` section.


In [5]:
from src.utils.config import load_config
from src.data.dataloaders import load_train_data, load_validation_data
from src.pipeline.baseline import run_baselines

config = load_config()
config["model"]


{'type': 'logistic_regression',
 'logistic_regression': {'C': [0.1, 1.0, 10.0],
  'penalty': ['l2'],
  'solver': ['lbfgs'],
  'max_iter': [1000]},
 'xgboost': {'n_estimators': [300],
  'max_depth': [4, 6, 8],
  'learning_rate': [0.05, 0.1],
  'subsample': [0.8],
  'colsample_bytree': [0.8],
  'objective': ['binary:logistic'],
  'tree_method': ['hist'],
  'eval_metric': ['logloss']},
 'baseline': {'model_type': ['linear_svm'],
  'C': [0.1, 1.0, 10.0],
  'n_estimators': [100],
  'max_depth': [10]}}

## 3. Run shared baselines on CSV data

This will:
- Load train and validation CSVs using the config-driven dataloaders.
- Train and evaluate the baseline models defined in `src/pipeline/baseline.py` (no BoW/TF-IDF preprocessing).

The output is a small accuracy summary in the notebook output.


In [6]:
run_baselines()

Loaded data from data/raw/csv (train-*.csv + movie_directors/movie_writers).
Training logistic_regression...
  config: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000}
  Validation accuracy (logistic_regression): 0.5138

Training xgboost...
  config: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'logloss'}
  Validation accuracy (xgboost): 0.5320

Training baseline...
  config: {'model_type': 'linear_svm', 'C': 0.1, 'n_estimators': 100, 'max_depth': 10}
  Validation accuracy (baseline): 0.5138

Summary (model -> config -> validation accuracy):
  logistic_regression: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000} -> 0.5138
  xgboost: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'logloss'} -> 0.5320
  b

## 4. Generate submission files

Run this after `run_baselines()`. Creates two text files for the course submission server (one line per row: `True` or `False`) with timestamp and best model name in the filename, in the `submissions/` folder.

In [ ]:
from src.pipeline.baseline import make_submission
make_submission()